# 04 梨叶分类论文与 MobileNetV2 结构

本讲整理我们对 CBAM 梨叶分类论文的讨论，并展开一个轻量骨干 MobileNetV2。目标是看懂：图片如何变成特征图、CBAM 接在哪里，以及论文怎样比较加入前后的效果。

论文：[Improving Deep Learning-based Plant Disease Classification with Attention Mechanism](https://doi.org/10.1007/s10343-022-00796-y)；[论文整体结构图（图 2）](https://link.springer.com/article/10.1007/s10343-022-00796-y/figures/2)。

## 1. 论文要解决什么问题

任务是**图像分类**：输入梨叶图片，判断为健康叶、叶斑、卷叶或蛞蝓危害四类中的一类，不要求预测病斑边界框。

田间图片可能受到光照、背景和拍摄角度影响；不同症状有时比较相似。作者希望利用 CBAM 调整 CNN 特征，改善分类效果。这是研究目标，不代表仅凭设计就能证明模型排除了背景干扰。

## 2. 论文的两组模型

对照组：图片 → 预训练 CNN → 分类头 → 四类预测。

实验组：图片 → 预训练 CNN → CBAM → 分类头 → 四类预测。

CBAM 接在 CNN 的**输出特征图之后**，并不是代替 CNN，也不是论文图 2 所示的每层之后都放一个。论文使用 MobileNetV2、EfficientNetB0 等多个骨干分别对照。论文文字提到分类部分包含全局平均池化、批量归一化、Flatten、Dense、Dropout 和 Softmax；精确的排列、维度与实现细节应进一步核对，不把下文的简化示意当作作者完整代码。

## 3. 什么是预训练 CNN

CNN 是提取图像特征的网络；“预训练”表示这些网络参数已经在其他数据集上学习过。论文使用 ImageNet 预训练模型，再用于梨叶四分类。

原始 RGB 输入通常为 $B \times 3 \times 224 \times 224$，经过卷积网络后变为多通道特征图。多出来的通道不是新的颜色，而是网络对输入的不同特征响应。

论文方法部分称作者冻结预训练 CNN 的卷积层；新加入的 CBAM 和分类部分参与学习。冻结卷积参数不自动保证 BatchNorm 的运行均值和方差不变，具体行为依赖框架和训练模式，复现时需要检查。

## 4. 以 MobileNetV2 展开 CNN 内部

下面是**本机 torchvision 标准 MobileNetV2** 的结构概览，输入 $B \times 3 \times 224 \times 224$。它用于理解论文采用的 MobileNetV2 骨干，不表示论文图 2 展示了以下每一层，也不保证作者 Keras 版本与 torchvision 完全一致。$B$ 是批次大小，表中省略它。

| 阶段 | 操作及重复次数 | 输出形状（通道 × 高 × 宽） |
|---|---|---|
| 输入 | RGB 图片 | $3 \times 224 \times 224$ |
| 初始卷积 | $3 \times 3$，步幅 2 | $32 \times 112 \times 112$ |
| 倒残差组 1 | 1 个块 | $16 \times 112 \times 112$ |
| 倒残差组 2 | 2 个块 | $24 \times 56 \times 56$ |
| 倒残差组 3 | 3 个块 | $32 \times 28 \times 28$ |
| 倒残差组 4 | 4 个块 | $64 \times 14 \times 14$ |
| 倒残差组 5 | 3 个块 | $96 \times 14 \times 14$ |
| 倒残差组 6 | 3 个块 | $160 \times 7 \times 7$ |
| 倒残差组 7 | 1 个块 | $320 \times 7 \times 7$ |
| 最后卷积 | $1 \times 1$ 卷积 | $1280 \times 7 \times 7$ |

上述组共有 17 个倒残差块。我们之前在本机使用 $224 \times 224$ 的模拟输入，实际观察到 torchvision 特征部分输出 $B \times 1280 \times 7 \times 7$。那只是形状与显存检查，没有使用梨叶数据训练。

## 5. 倒残差块在做什么

MobileNetV2 的常见倒残差块可概括为：

$$\text{输入} \rightarrow 1\times1\text{ 卷积扩展通道} \rightarrow 3\times3\text{ 逐通道卷积} \rightarrow 1\times1\text{ 卷积压缩通道} \rightarrow \text{输出}$$

扩展后的通道分别做空间卷积，再通过 $1 \times 1$ 卷积组合并压缩通道。逐通道卷积每个输入通道单独进行空间卷积，是轻量化设计的一部分。

当步幅为 1 且输入、输出通道数一致时，块会把输入与变换后的输出相加，形成残差连接；形状不同则不能直接相加。第一个块的扩展倍数为 1，所以 torchvision 实现中省略了单独的扩展卷积。原理详见 [MobileNetV2 论文](https://arxiv.org/abs/1801.04381)。

## 6. 在特征输出后加入 CBAM

以本机 torchvision 的标准特征输出为例，可用以下形状理解论文的接入位置：

$$B \times 3 \times 224 \times 224 \rightarrow \text{MobileNetV2 特征提取} \rightarrow B \times 1280 \times 7 \times 7$$

接着经过 CBAM：

1. 通道注意力生成 $B \times 1280 \times 1 \times 1$ 的权重，乘回特征。
2. 空间注意力生成 $B \times 1 \times 7 \times 7$ 的权重，乘回上一步结果。
3. CBAM 输出仍为 $B \times 1280 \times 7 \times 7$，再送给分类部分。

如果接全局平均池化，将每个通道的 $7 \times 7$ 概括成一个数，则得到 $B \times 1280$；最终四分类头输出 $B \times 4$ 的类别分数。这里是**形状示例**，不等于已完整确认论文分类头的全部层和参数。训练使用交叉熵时可直接输入 logits；推理时可使用 Softmax 转为类别概率。

## 7. 两种注意力与自注意力

CBAM 的通道注意力先对高和宽分别做平均与最大池化，再通过共享 MLP、相加和 Sigmoid，生成每个通道的权重。空间注意力沿通道做平均与最大池化，拼接后卷积并经过 Sigmoid，得到每个空间位置的权重。两者都将权重乘回特征。

这些权重依赖当前输入；MLP 和空间卷积的参数在训练中根据任务损失，通过反向传播和优化器更新。冻结骨干卷积参数不会自动冻结 CBAM。

CBAM 不使用 Transformer 式 Q、K、V 自注意力。我们之前做过串联通道与空间 QKV 自注意力的探索变体，它会混合不同通道或位置的信息，**不是论文原始 CBAM，也不是论文的实验结果**。

## 8. 数据、对照与结果

论文使用 DiaMOS Plant 梨叶图片，包含健康叶 43 张、叶斑 884 张、卷叶 54 张、蛞蝓危害 2025 张，共 3006 张梨叶图片；类别不均衡。[数据集](https://zenodo.org/records/5557313)还包含果实图片，下载压缩包约 13.1 GB。

论文作者报告原有数据划分的训练与测试图片有重叠，因而重新采用十折交叉验证。比较有无 CBAM 时，应保持数据划分和其他设置尽量相同。仅避免同一图片重叠仍不能保证同一棵树的相似图片不会分别进入训练和测试。

| 论文骨干 | 不加 CBAM 的平均准确率 | 加 CBAM 的平均准确率 | 差值 |
|---|---:|---:|---:|
| MobileNetV2 | 82.06% | 83.99% | 1.93 个百分点 |
| EfficientNetB0 | 85.82% | 86.89% | 1.07 个百分点 |

这些是论文报告的十折平均结果，不是我们本机的实验。最终准确率最高与加入 CBAM 后提升最多是不同问题。

## 9. 如何判断结果与限制

- 准确率是全部样本中的正确比例；类别不均衡时还应看各类召回率、精确率和混淆矩阵。
- 某类召回率：实际属于该类的样本中，有多少被正确找出。某类精确率：预测为该类的样本中，有多少确实属于该类。
- 论文使用 Grad-CAM 辅助观察预测相关区域。Grad-CAM 不是 CBAM 的空间权重图；热力图不能单独证明模型的因果判断过程。
- 图片来自同一地块的三棵树，若要评价跨果园泛化，还需要独立的新果园数据。
- 我们的设备做过随机输入的冻结骨干试运行，batch size 32 可完成前向与反向；这不证明真实数据训练一定获得论文结果或能达到论文速度。

## 10. 小结与自测

1. 原始 RGB 图像的 3 个通道和 CNN 输出的 1280 个特征通道有什么区别？
2. MobileNetV2 常见倒残差块如何扩展、处理和压缩通道？何时有残差连接？
3. 若使用本机标准 torchvision MobileNetV2，$224 \times 224$ 输入经过特征提取后形状是什么？CBAM 改变形状吗？
4. 论文比较有无 CBAM 时，为什么要保持骨干和数据划分一致？
5. 为什么整体准确率提高不一定代表每一类召回率都提高？
6. 为什么论文结果和我们先前的 CIFAR-10 自注意力实验不能直接比较？

参考：[论文全文](https://doi.org/10.1007/s10343-022-00796-y)；[论文结构图](https://link.springer.com/article/10.1007/s10343-022-00796-y/figures/2)；[Torchvision MobileNetV2](https://docs.pytorch.org/vision/main/models/generated/torchvision.models.mobilenet_v2.html)。